In [1]:
#1:

import os
from pathlib import Path

import requests

from urllib.parse import urljoin, urlparse
from tenacity import retry, stop_after_attempt, wait_fixed

from langchain_core.documents import Document



from langchain_community.document_loaders import (
    PyMuPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader,
    UnstructuredHTMLLoader,
    UnstructuredMarkdownLoader,
    UnstructuredXMLLoader,
    UnstructuredImageLoader,
    UnstructuredEmailLoader,
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

import requests
import tempfile
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

/var/folders/ln/crlsn70x6sngf7xb35d48hr80000gn/T/ipykernel_1819/1378935031.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/Users/ankittehri/Desktop/RAG_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#2:

LOADER_MAPPING = {

    ".pdf": PyMuPDFLoader,

    ".docx": UnstructuredWordDocumentLoader,
    ".doc": UnstructuredWordDocumentLoader,

    ".pptx": UnstructuredPowerPointLoader,
    ".ppt": UnstructuredPowerPointLoader,

    ".xlsx": UnstructuredExcelLoader,
    ".xls": UnstructuredExcelLoader,

    ".csv": CSVLoader,

    ".txt": TextLoader,

    ".html": UnstructuredHTMLLoader,
    ".htm": UnstructuredHTMLLoader,

    ".xml": UnstructuredXMLLoader,

    ".md": UnstructuredMarkdownLoader,

    ".png": UnstructuredImageLoader,
    ".jpg": UnstructuredImageLoader,
    ".jpeg": UnstructuredImageLoader,

    ".eml": UnstructuredEmailLoader,
    ".msg": UnstructuredEmailLoader,

    ".json": TextLoader,
    ".yaml": TextLoader,
    ".yml": TextLoader,
    ".rtf": TextLoader,
}

In [3]:
#3:

SUPPORTED_EXTENSIONS = tuple(LOADER_MAPPING.keys())


def is_same_domain(base_url, target_url):

    return urlparse(base_url).netloc == urlparse(target_url).netloc

In [4]:
#4:

MAX_CRAWL_DEPTH = 3

MAX_PAGES = 500

REQUEST_TIMEOUT = 30

USER_AGENT = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}

visited_urls = set()

visited_documents = set()

In [5]:
#5:

@retry(
    stop=stop_after_attempt(3),
    wait=wait_fixed(2)
)
def fetch_page(url):

    response = requests.get(
        url,
        headers=USER_AGENT,
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    return response

In [6]:
#6:

def extract_links(base_url, html):

    soup = BeautifulSoup(html, "html.parser")

    page_links = set()
    document_links = set()

    for tag in soup.find_all("a", href=True):

        href = tag.get("href", "").strip()

        if not href:
            continue

        # Ignore invalid links
        if href.startswith((
            "#",
            "javascript:",
            "mailto:",
            "tel:"
        )):
            continue

        absolute_url = urljoin(base_url, href)

        parsed = urlparse(absolute_url)

        clean_url = parsed._replace(
            fragment=""
        ).geturl()

        # Crawl only same domain
        if not is_same_domain(base_url, clean_url):
            continue

        extension = Path(parsed.path).suffix.lower()

        if extension in SUPPORTED_EXTENSIONS:
            document_links.add(clean_url)
        else:
            page_links.add(clean_url)

    return page_links, document_links

In [7]:
#7:

from collections import deque

def crawl_website(start_url):

    queue = deque()

    queue.append((start_url, 0))

    pages = []

    documents = set()

    visited_urls.clear()

    while queue:

        current_url, depth = queue.popleft()

        if current_url in visited_urls:
            continue

        if depth > MAX_CRAWL_DEPTH:
            continue

        if len(visited_urls) >= MAX_PAGES:
            break

        try:

            response = fetch_page(current_url)

            visited_urls.add(current_url)

            pages.append(
                {
                    "url": current_url,
                    "html": response.text
                }
            )

            page_links, document_links = extract_links(
                current_url,
                response.text
            )

            documents.update(document_links)

            for link in page_links:

                if link not in visited_urls:

                    queue.append(
                        (
                            link,
                            depth + 1
                        )
                    )

            print(f"✓ {current_url}")

        except Exception as e:

            print(f"✗ {current_url}")
            print(e)

    return pages, list(documents)

In [8]:
#8:

pages, docs = crawl_website(
    "https://python.langchain.com"
)

print(f"Pages Crawled: {len(pages)}")
print(f"Documents Found: {len(docs)}")

✓ https://python.langchain.com
✓ https://python.langchain.com/oss/python/langchain/observability
✓ https://python.langchain.com/oss/python/langchain/agents
✓ https://python.langchain.com/oss/python/integrations/providers/overview
✓ https://python.langchain.com/oss/python/langchain/install
✓ https://python.langchain.com/oss/python/langchain/middleware/built-in
✓ https://python.langchain.com/langsmith/observability
✓ https://python.langchain.com/oss/python/langchain/structured-output
✓ https://python.langchain.com/use-these-docs
✓ https://python.langchain.com/oss/python/langchain/guardrails
✓ https://python.langchain.com/oss/python/contributing/overview
✓ https://python.langchain.com/oss/python/reference/overview
✓ https://python.langchain.com/oss/python/learn
✓ https://python.langchain.com/oss/python/langgraph/overview
✓ https://python.langchain.com/langsmith/engine
✓ https://python.langchain.com/oss/python/langchain/short-term-memory
✓ https://python.langchain.com/oss/python/langchain/

# Convert Crawled Webpages to LangChain Documents

This step converts the crawled HTML pages into LangChain `Document`
objects while preserving important metadata required for retrieval.

Metadata stored:
- source_url
- domain
- page_title
- crawl_time
- content_type

In [9]:
#9:

from datetime import datetime
from langchain_core.documents import Document


def convert_pages_to_documents(pages):

    documents = []

    crawl_time = datetime.utcnow().isoformat()

    for page in pages:

        soup = BeautifulSoup(page["html"], "html.parser")

        title = soup.title.string.strip() if soup.title else "Untitled"

        # Remove unnecessary tags
        for tag in soup(["script", "style", "noscript", "header", "footer", "svg"]):
            tag.decompose()

        text = soup.get_text(separator="\n", strip=True)

        if not text:
            continue

        document = Document(
            page_content=text,
            metadata={
                "source_url": page["url"],
                "domain": urlparse(page["url"]).netloc,
                "page_title": title,
                "crawl_time": crawl_time,
                "content_type": "website",
            },
        )

        documents.append(document)

    print(f"Converted {len(documents)} webpages into LangChain Documents.")

    return documents

In [10]:
#10:

website_documents = convert_pages_to_documents(pages)

print("=" * 60)
print(f"Total Website Documents : {len(website_documents)}")
print("=" * 60)

print("\nMetadata\n")
print(website_documents[0].metadata)

print("\nContent Preview\n")
print(website_documents[0].page_content[:500])

/var/folders/ln/crlsn70x6sngf7xb35d48hr80000gn/T/ipykernel_1819/718072690.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  crawl_time = datetime.utcnow().isoformat()


Converted 40 webpages into LangChain Documents.
Total Website Documents : 40

Metadata

{'source_url': 'https://python.langchain.com', 'domain': 'python.langchain.com', 'page_title': 'LangChain overview - Docs by LangChain', 'crawl_time': '2026-07-12T08:58:15.039747', 'content_type': 'website'}

Content Preview

LangChain overview - Docs by LangChain
Documentation Index
Fetch the complete documentation index at:
/llms.txt
Use this file to discover all available pages before exploring further.
Skip to main content
Python
Overview
Get started
Install
Quickstart
Changelog
Philosophy
Core components
Agents
Models
Messages
Tools
Short-term memory
Event streaming
Streaming
Structured output
Middleware
Overview
Prebuilt middleware
Custom middleware
Frontend
Overview
Patterns
Integrations
Advanced usage
Guardra


In [11]:
#11:

def process_document_url(url):

    extension = Path(urlparse(url).path).suffix.lower()

    if extension not in LOADER_MAPPING:

        return []

    if url in visited_documents:

        return []

    try:

        response = requests.get(
            url,
            headers=USER_AGENT,
            timeout=REQUEST_TIMEOUT,
        )

        response.raise_for_status()

        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=extension
        ) as temp:

            temp.write(response.content)

            temp_path = temp.name

        loader = LOADER_MAPPING[extension](temp_path)

        documents = loader.load()

        for document in documents:

            document.metadata.update({

                "source_url": url,

                "domain": urlparse(url).netloc,

                "file_name": Path(urlparse(url).path).name,

                "file_type": extension,

                "content_type": "document"

            })

        visited_documents.add(url)

        print(f"✓ Loaded : {url}")

        return documents

    except Exception as e:

        print(f"✗ Failed : {url}")

        print(e)

        return []

    finally:

        try:

            os.remove(temp_path)

        except:

            pass

In [12]:
# TEST CELL / 12:

all_downloaded_documents = []

for document_url in docs:

    all_downloaded_documents.extend(
        process_document_url(document_url)
    )

print()

print("="*60)

print(f"Downloaded Documents : {len(all_downloaded_documents)}")

print("="*60)

✓ Loaded : https://python.langchain.com/llms.txt

Downloaded Documents : 1


In [13]:
#13:

def merge_documents(website_documents, downloaded_documents):

    merged_documents = []

    seen_content = set()

    all_documents = website_documents + downloaded_documents

    for document in all_documents:

        if document is None:
            continue

        if not hasattr(document, "page_content"):
            continue

        text = document.page_content.strip()

        if len(text) < 20:
            continue

        content_hash = hash(text)

        if content_hash in seen_content:
            continue

        seen_content.add(content_hash)

        metadata = document.metadata.copy()

        metadata.setdefault("source_url", "Unknown")
        metadata.setdefault("domain", "Unknown")
        metadata.setdefault("content_type", "Unknown")

        document.metadata = metadata

        merged_documents.append(document)

    print(f"Final Documents : {len(merged_documents)}")

    return merged_documents

In [14]:
# TEST CELL / 14:

documents = merge_documents(
    website_documents,
    all_downloaded_documents
)

print()

print("=" * 60)

print(f"Total Final Documents : {len(documents)}")

print("=" * 60)

print()

print(documents[0].metadata)

Final Documents : 2

Total Final Documents : 2

{'source_url': 'https://python.langchain.com', 'domain': 'python.langchain.com', 'page_title': 'LangChain overview - Docs by LangChain', 'crawl_time': '2026-07-12T08:58:15.039747', 'content_type': 'website'}


In [15]:
#15:

import uuid


def create_chunks(documents):

    splitter = RecursiveCharacterTextSplitter(

        chunk_size=1000,

        chunk_overlap=200,

        separators=[

            "\n\n",

            "\n",

            ". ",

            "? ",

            "! ",

            " ",

            ""

        ]
    )

    chunks = splitter.split_documents(documents)

    for index, chunk in enumerate(chunks):

        chunk.metadata["chunk_id"] = str(uuid.uuid4())

        chunk.metadata["chunk_number"] = index + 1

    print(f"Created {len(chunks)} chunks.")

    return chunks

In [16]:
# TEST CELL/ 16:

chunks = create_chunks(documents)

print()

print("=" * 60)

print(f"Total Chunks : {len(chunks)}")

print("=" * 60)

print()

print(chunks[0].metadata)

print()

print(chunks[0].page_content[:500])

Created 138 chunks.

Total Chunks : 138

{'source_url': 'https://python.langchain.com', 'domain': 'python.langchain.com', 'page_title': 'LangChain overview - Docs by LangChain', 'crawl_time': '2026-07-12T08:58:15.039747', 'content_type': 'website', 'chunk_id': 'b24161e5-c596-46bb-bfdc-98c6cac4a7d0', 'chunk_number': 1}

LangChain overview - Docs by LangChain
Documentation Index
Fetch the complete documentation index at:
/llms.txt
Use this file to discover all available pages before exploring further.
Skip to main content
Python
Overview
Get started
Install
Quickstart
Changelog
Philosophy
Core components
Agents
Models
Messages
Tools
Short-term memory
Event streaming
Streaming
Structured output
Middleware
Overview
Prebuilt middleware
Custom middleware
Frontend
Overview
Patterns
Integrations
Advanced usage
Guardra


In [17]:
#17:

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding Model Loaded Successfully.")

/var/folders/ln/crlsn70x6sngf7xb35d48hr80000gn/T/ipykernel_1819/2999627372.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6666.24it/s]


Embedding Model Loaded Successfully.


In [18]:
# TEST CELL/ 18:

query_embedding = embeddings.embed_query(
    "What is LangChain?"
)

print(f"Embedding Dimension : {len(query_embedding)}")

Embedding Dimension : 384


In [19]:
#19:

VECTOR_DB_ROOT = "./vector_db"

WORKSPACE_NAME = "default"

PROJECT_NAME = urlparse(
    documents[0].metadata["source_url"]
).netloc.replace(".", "_")

COLLECTION_NAME = f"{WORKSPACE_NAME}_{PROJECT_NAME}"

VECTOR_DB_PATH = os.path.join(
    VECTOR_DB_ROOT,
    COLLECTION_NAME
)

os.makedirs(
    VECTOR_DB_PATH,
    exist_ok=True
)

vectorstore = Chroma(

    collection_name=COLLECTION_NAME,

    embedding_function=embeddings,

    persist_directory=VECTOR_DB_PATH

)

print()

print("=" * 60)

print("Vector Database Initialized Successfully.")

print(f"Collection : {COLLECTION_NAME}")

print("=" * 60)

/var/folders/ln/crlsn70x6sngf7xb35d48hr80000gn/T/ipykernel_1819/3442270095.py:23: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(



Vector Database Initialized Successfully.
Collection : default_python_langchain_com


In [20]:
#20:

import hashlib
from datetime import datetime, timezone


def generate_sha256(text):

    return hashlib.sha256(

        text.encode("utf-8")

    ).hexdigest()


def enrich_document_metadata(documents):

    ingestion_time = datetime.now(
        timezone.utc
    ).isoformat()

    for document in documents:

        source = document.metadata.get(
            "source_url",
            "unknown"
        )

        document.metadata["document_id"] = generate_sha256(
            source + document.page_content
        )

        document.metadata["content_hash"] = generate_sha256(
            document.page_content
        )

        document.metadata["document_size"] = len(
            document.page_content
        )

        document.metadata["language"] = "unknown"

        document.metadata["processing_status"] = "processed"

        document.metadata["ingestion_time"] = ingestion_time

    print(
        f"Metadata enriched for {len(documents)} documents."
    )

    return documents

In [21]:
#21:

documents = enrich_document_metadata(documents)

print(documents[0].metadata)

Metadata enriched for 2 documents.
{'source_url': 'https://python.langchain.com', 'domain': 'python.langchain.com', 'page_title': 'LangChain overview - Docs by LangChain', 'crawl_time': '2026-07-12T08:58:15.039747', 'content_type': 'website', 'document_id': 'b1795cec0a1df67cb7939e4483a080d242e95b9fbaf28952c5338d9de55213b3', 'content_hash': '8eafe391a01c1b1c101fde495042fc73b6b15de525d7952f5dda1fa369a5c3ce', 'document_size': 8799, 'language': 'unknown', 'processing_status': 'processed', 'ingestion_time': '2026-07-12T08:58:23.751271+00:00'}


In [22]:
#22:

INVALID_PATTERNS = [

    "404",

    "page not found",

    "access denied",

    "forbidden",

    "login",

    "sign in",

    "sign up",

    "javascript is disabled",

    "cookie policy",

    "enable javascript",

    "captcha",

    "robot check",

    "cloudflare",

    "service unavailable",

    "internal server error"

]


def validate_documents(documents):

    validated = []

    seen_hashes = set()

    for document in documents:

        text = document.page_content.strip()

        if len(text.split()) < 10:
            continue

        lower = text.lower()

        if any(
            pattern in lower
            for pattern in INVALID_PATTERNS
        ):
            continue

        content_hash = document.metadata.get(
            "content_hash"
        )

        if content_hash in seen_hashes:
            continue

        seen_hashes.add(content_hash)

        validated.append(document)

    print(
        f"Validated Documents : {len(validated)}"
    )

    return validated

In [23]:
#23:

vectorstore.add_documents(
    chunks
)

print()

print("=" * 60)

print(
    f"Stored {len(chunks)} chunks in Chroma."
)

print("=" * 60)


Stored 138 chunks in Chroma.


In [24]:
#24:


retriever = vectorstore.as_retriever(

    search_type="similarity",

    search_kwargs={

        "k": 5

    }

)

print()

print("=" * 60)

print("Retriever Created Successfully.")

print("=" * 60)


Retriever Created Successfully.


In [ ]:
# Test Cell/ 25:

results = retriever.invoke(
    "What is LangChain?"
)

print()

print("=" * 60)

print(f"Retrieved {len(results)} chunks.")

print("=" * 60)

for i, doc in enumerate(results, 1):

    print(f"\nResult {i}")

    print("-" * 40)

    print(doc.metadata)

    print()

    print(doc.page_content[:500])


Retrieved 5 chunks.

Result 1
----------------------------------------
{'crawl_time': '2026-07-11T09:40:55.295453', 'chunk_number': 11, 'source_url': 'https://python.langchain.com', 'chunk_id': '5ef374a6-06f3-4286-b713-74f8aff4a58e', 'page_title': 'LangChain overview - Docs by LangChain', 'content_type': 'website', 'domain': 'python.langchain.com'}

Learn more
Highly configurable harness
Start with
create_agent
as a minimal harness and add capabilities incrementally through middleware. Compose only what your use case needs, from guardrails and retries to routing and custom tool policies.
Learn more
Built on top of LangGraph
LangChain’s agents are built on top of LangGraph. This allows us to take advantage of LangGraph’s durable execution, human-in-the-loop support, persistence, and more.
Learn more
Debug with LangSmith
Inspect traces, tool 

Result 2
----------------------------------------
{'page_title': 'LangChain overview - Docs by LangChain', 'domain': 'python.langchain.com', 'chu

In [26]:
#26:

from dotenv import load_dotenv
import os

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("❌ GOOGLE_API_KEY not found in .env")

LLM_MODEL = "gemini-3.5-flash"

llm = ChatGoogleGenerativeAI(

    model=LLM_MODEL,

    google_api_key=GOOGLE_API_KEY,

    temperature=0

)

print()

print("=" * 60)

print("✅ Gemini LLM Loaded Successfully.")

print("=" * 60)


✅ Gemini LLM Loaded Successfully.


In [27]:
#27:

from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """
You are an intelligent AI assistant that answers questions using ONLY the retrieved website context.

Rules:

1. Use ONLY the provided context to answer.
2. Never use your own knowledge if the answer is missing.
3. Never hallucinate or make assumptions.
4. If the answer is not present in the context, respond exactly:
   "I couldn't find that information in the indexed website."
5. Keep the answer clear, concise and well-structured.
6. If multiple pieces of context are relevant, combine them into a single answer.
7. Do not mention embeddings, vector databases, retrieval systems or internal implementation.
8. If the context contains conflicting information, mention that politely.
"""

prompt = ChatPromptTemplate.from_messages(

    [

        (

            "system",

            SYSTEM_PROMPT

        ),

        (

            "human",

            """
Context:

{context}

Question:

{input}
"""

        )

    ]

)

print()

print("=" * 60)

print("Production Prompt Created Successfully.")

print("=" * 60)


Production Prompt Created Successfully.


In [28]:
#28:

retriever = vectorstore.as_retriever(

    search_type="similarity",

    search_kwargs={

        "k": 5

    }

)

print()

print("=" * 60)

print("Retriever Created Successfully.")

print("=" * 60)


Retriever Created Successfully.


In [29]:
#29: Test Cell

query = "What is LangChain?"

retrieved_docs = retriever.invoke(query)

print()

print("=" * 60)

print(f"Retrieved {len(retrieved_docs)} Documents")

print("=" * 60)

print()

print(retrieved_docs[0].page_content[:700])


Retrieved 5 Documents

Learn more
Highly configurable harness
Start with
create_agent
as a minimal harness and add capabilities incrementally through middleware. Compose only what your use case needs, from guardrails and retries to routing and custom tool policies.
Learn more
Built on top of LangGraph
LangChain’s agents are built on top of LangGraph. This allows us to take advantage of LangGraph’s durable execution, human-in-the-loop support, persistence, and more.
Learn more
Debug with LangSmith
Inspect traces, tool calls, state transitions, and latency in one place. Find failure modes, evaluate quality, and improve agent behavior with execution data.
Learn more
Connect these docs
to Claude, VSCode, and more via 


In [39]:
#30: Debug Retrieval

results = retriever.invoke("agents")

for i, doc in enumerate(results, 1):

    print("=" * 60)
    print(f"Result {i}")
    print("=" * 60)

    print("Title :", doc.metadata.get("page_title"))

    print("URL   :", doc.metadata.get("source_url"))

    print()

    print(doc.page_content[:500])

    print("\n")

Result 1
Title : LangChain overview - Docs by LangChain
URL   : https://python.langchain.com

Agent development
LangSmith Studio
Test
Agent Chat UI
Production
Deployment
Observability
On this page
Create an agent
Core benefits
Agent = Model + Harness.
LangChain provides
create_agent
: a minimal, highly configurable harness. The harness is everything around the model loop: the prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what your use case needs. Supports
OpenAI, Anthropic, Google, and more
.
LangChain vs. LangGraph vs. Deep Age


Result 2
Title : LangChain overview - Docs by LangChain
URL   : https://python.langchain.com

Agent development
LangSmith Studio
Test
Agent Chat UI
Production
Deployment
Observability
On this page
Create an agent
Core benefits
Agent = Model + Harness.
LangChain provides
create_agent
: a minimal, highly configurable harness. The harness is everything around the model loop: the prompt, the tools, and a

In [ ]:
#31:

def format_docs(docs):

    return "\n\n".join(

        doc.page_content

        for doc in docs

    )

print()

print("=" * 60)

print("Context Formatter Created Successfully.")

print("=" * 60)


Context Formatter Created Successfully.


In [ ]:
#32:

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (

    {

        "context": retriever | format_docs,

        "input": RunnablePassthrough()

    }

    | prompt

    | llm

    | StrOutputParser()

)

print()

print("=" * 60)

print("Production RAG Chain Created Successfully.")

print("=" * 60)


Production RAG Chain Created Successfully.


In [41]:
#33: Test Cell:

question = "What is LangSmith?"

response = rag_chain.invoke(

    question

)

print()

print("=" * 60)

print("Answer")

print("=" * 60)

print()

print(response)


Answer

I couldn't find that information in the indexed website.


In [ ]:
#34:

def calculate_confidence(question):

    results = vectorstore.similarity_search_with_score(

        question,

        k=5

    )

    if not results:

        return 0.0

    similarities = []

    for _, score in results:

        similarity = 1 / (1 + score)

        similarities.append(similarity)

    confidence = (

        sum(similarities)

        / len(similarities)

    ) * 100

    return round(confidence, 2)

In [ ]:
#35:

def ask_question(question):

    retrieved_docs = retriever.invoke(question)

    answer = rag_chain.invoke(question)

    # TEMPORARILY DISABLED
    confidence = 100

    sources = []

    seen_sources = set()

    for doc in retrieved_docs:

        source_url = doc.metadata.get("source_url", "Unknown")

        if source_url in seen_sources:
            continue

        seen_sources.add(source_url)

        sources.append(
            {
                "title": doc.metadata.get("page_title", "Unknown"),
                "url": source_url,
                "content_type": doc.metadata.get("content_type", "Unknown")
            }
        )

    return {
        "question": question,
        "answer": answer,
        "confidence": confidence,
        "sources": sources,
        "total_sources": len(sources)
    }

In [42]:
#36:

question = input(

    "Ask a Question: "

)

response = ask_question(

    question

)

print()

print("=" * 60)

print("ANSWER")

print("=" * 60)

print(response["answer"])

print()

print("=" * 60)

print("CONFIDENCE")

print("=" * 60)

print(f"{response['confidence']} %")

print()

print("=" * 60)

print(f"SOURCES ({response['total_sources']})")

print("=" * 60)

for source in response["sources"]:

    print(

        f"Title : {source['title']}"

    )

    print(

        f"URL   : {source['url']}"

    )

    print(

        f"Type  : {source['content_type']}"

    )

    print("-" * 60)


ANSWER
I couldn't find that information in the indexed website.

CONFIDENCE
100 %

SOURCES (2)
Title : Unknown
URL   : https://python.langchain.com/llms.txt
Type  : document
------------------------------------------------------------
Title : LangChain overview - Docs by LangChain
URL   : https://python.langchain.com
Type  : website
------------------------------------------------------------


In [43]:
#37:


from urllib.parse import urljoin


def detect_llms_txt(base_url):

    llms_url = urljoin(

        base_url,

        "/llms.txt"

    )

    try:

        response = requests.get(

            llms_url,

            headers=USER_AGENT,

            timeout=10

        )

        if response.status_code == 200:

            print()

            print("=" * 60)

            print("✅ llms.txt Found")

            print("=" * 60)

            print(llms_url)

            return llms_url

        print()

        print("=" * 60)

        print("❌ llms.txt Not Found")

        print("=" * 60)

        return None

    except Exception as e:

        print()

        print("=" * 60)

        print("❌ Error Checking llms.txt")

        print("=" * 60)

        print(e)

        return None


llms_txt_url = detect_llms_txt(

    "https://python.langchain.com"

)


✅ llms.txt Found
https://python.langchain.com/llms.txt


In [46]:
#38:


def get_crawl_strategy(base_url):

    print()

    print("=" * 60)

    print("Selecting Crawl Strategy...")

    print("=" * 60)

    llms_url = detect_llms_txt(base_url)

    if llms_url:

        print()

        print("Strategy Selected : LLM Documentation Index")

        return {

            "strategy": "llms_txt",

            "llms_url": llms_url

        }

    print()

    print("Strategy Selected : Standard Website Crawl")

    return {

        "strategy": "crawler",

        "start_url": base_url

    }


crawl_strategy = get_crawl_strategy(

    "https://python.langchain.com"

)

print()

print("=" * 60)

print("Selected Strategy")

print("=" * 60)

print(crawl_strategy)


Selecting Crawl Strategy...

✅ llms.txt Found
https://python.langchain.com/llms.txt

Strategy Selected : LLM Documentation Index

Selected Strategy
{'strategy': 'llms_txt', 'llms_url': 'https://python.langchain.com/llms.txt'}
